<a href="https://colab.research.google.com/github/Qalani/Dissertation/blob/main/Batch_Export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Winam Gulf Sentinel predictor export notebook — snapshot-validity gated version

This notebook exports **single-date Sentinel-1/Sentinel-2 predictor snapshots** for the Winam Gulf water-hyacinth workflow.

It is designed specifically to avoid the problem where near-empty Sentinel-2 images are exported as large GeoTIFFs filled with `-9999`.

Key changes from the previous resumable notebook:

- keeps **single-date / one-day snapshot exports** rather than monthly composites;
- calculates valid-pixel coverage **after S2/S1 masking and after all predictor bands are built**;
- skips exports with too little valid coverage over the Winam water mask;
- writes skipped rows to a QA log so the notebook can resume cleanly;
- writes `-9999` as a proper GeoTIFF NoData value using `formatOptions`;
- uses a separate validated export folder by default to avoid mixing old invalid exports with clean exports.

Recommended use: run the notebook repeatedly. Each run queues the next batch of valid exports and records skipped low-coverage dates.


## 1. Imports

In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
from IPython.display import display
import csv
import json
import math

import ee
import geemap
import pandas as pd


## 2. Drive, Earth Engine, AOI, and export settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------------------------------------------
# Drive / Earth Engine setup
# -------------------------------------------------------------------

# Use a separate folder so clean, coverage-gated exports are not mixed
# with earlier one-day exports that may contain almost no valid pixels.
# Change this back to 'GEE_Exports' only if you intentionally want to
# write into the old export folder.
EE_EXPORT_FOLDER = 'GEE_Exports_validated_snapshots'
GEE_EXPORT_DIR = Path('/content/drive/MyDrive') / EE_EXPORT_FOLDER
GEE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

EE_PROJECT = 'ee-bmillwardsadler1'
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

# Adjusted Winam Gulf AOI.
winam = ee.Geometry.Rectangle([34, -0.55, 34.9, 0], geodesic=False)

# -------------------------------------------------------------------
# Export settings
# -------------------------------------------------------------------

EXPORT_CRS = 'EPSG:32736'
EXPORT_SCALE = 10
PREDICTOR_NODATA_VALUE = -9999.0

# Earth Engine tile dimensions. 8192 explains filenames such as:
# prefix-0000000000-0000000000.tif
# prefix-0000000000-0000008192.tif
FILE_DIMENSIONS = 8192

# When we unmask to -9999 for explicit NoData export, tiles are no longer
# "fully masked", so skipEmptyTiles is not very useful. Leave as None.
SKIP_EMPTY_TILES = None

# Batch exports are ON by default. Set False to only build/inspect manifests.
RUN_EXPORTS = True

# Sensor switches.
EXPORT_S2 = True
EXPORT_S1 = True

# Archive range. The manifest only includes dates with imagery.
S2_START_DATE = '2017-03-28'
S1_START_DATE = '2014-10-03'
EXPORT_END_DATE = date.today().isoformat()

# Sentinel-2 granule-level metadata cloud filter.
# This is only a coarse prefilter; the true AOI valid-pixel gate below
# determines whether each date is actually exported.
S2_CLOUD_PCT = 70

# Optional JRC water mask used in both export masking and coverage denominator.
APPLY_JRC_WATER_MASK = True
JRC_OCCURRENCE_THRESHOLD = 5

# -------------------------------------------------------------------
# Valid-pixel coverage gates
# -------------------------------------------------------------------

# For mobile WH mats, these are still single-date snapshots.
# The gate simply prevents near-empty snapshots from being exported.
VALIDATE_BEFORE_EXPORT = True

# Coverage denominator: 'water' is recommended for WH area mapping.
# It measures valid pixels as a fraction of the JRC water mask in Winam.
VALID_COVERAGE_DENOMINATOR = 'water'  # 'water' or 'aoi'

# Thresholds are deliberately strict for S2 because cloud/mask collapse is common.
# Adjust after looking at the QA manifest.
S2_MIN_VALID_FRACTION = 0.50
S2_MIN_VALID_PIXELS = 250_000

# S1 is not affected by cloud, but the gate catches partial swath/processing issues.
S1_MIN_VALID_FRACTION = 0.50
S1_MIN_VALID_PIXELS = 250_000

# -------------------------------------------------------------------
# Resumable batch controls
# -------------------------------------------------------------------

MAX_EXPORTS_PER_RUN = 100
MAX_ACTIVE_TASKS = 125

AUTO_RESUME_FROM_PROGRESS = True
SKIP_PREFIXES_ALREADY_PROCESSED_IN_PROGRESS = True
SKIP_PREFIXES_ALREADY_IN_DRIVE = True

# Manual override. Leave as None for automatic resume.
MANUAL_START_AT_MANIFEST_ROW = None

# Set True once to delete the progress JSON and start queueing from the beginning.
# Existing TIFFs will still be skipped when SKIP_PREFIXES_ALREADY_IN_DRIVE=True.
RESET_EXPORT_PROGRESS = False

PROGRESS_PATH = GEE_EXPORT_DIR / 'winam_snapshot_validated_predictor_export_progress.json'
EXPORT_EVENT_LOG_PATH = GEE_EXPORT_DIR / 'winam_snapshot_validated_predictor_export_event_log.csv'

print('Earth Engine project:', EE_PROJECT)
print('Drive export folder:', GEE_EXPORT_DIR)
print('S2 date range:', S2_START_DATE, 'to', EXPORT_END_DATE)
print('S1 date range:', S1_START_DATE, 'to', EXPORT_END_DATE)
print('RUN_EXPORTS:', RUN_EXPORTS)
print('VALIDATE_BEFORE_EXPORT:', VALIDATE_BEFORE_EXPORT)
print('Coverage denominator:', VALID_COVERAGE_DENOMINATOR)
print('S2 gate:', S2_MIN_VALID_PIXELS, 'pixels and', f'{S2_MIN_VALID_FRACTION:.0%}', 'coverage')
print('S1 gate:', S1_MIN_VALID_PIXELS, 'pixels and', f'{S1_MIN_VALID_FRACTION:.0%}', 'coverage')
print('Progress JSON:', PROGRESS_PATH)
print('Event log:', EXPORT_EVENT_LOG_PATH)


## 3. Predictor band definitions

In [ ]:
S2_PREDICTORS = [
    'AWEI_p95', 'AWEI', 'AWEInsh', 'NDMI', 'MNDWI', 'NDVI',
    'B', 'G', 'R', 'RE1', 'RE2', 'RE3', 'RE4', 'NIR', 'SWIR', 'SWIR2'
]

S1_REF_ANGLE_DEGREES = 38.0
S1_PREDICTORS = ['VH_p5', 'VH_corrected', 'VH_smooth']

# Baseline periods copied from the existing Route B workflow.
S2_AWEI_P95_START = '2017-04-01'
S2_AWEI_P95_END = '2021-06-01'
S1_VH_P5_START = '2015-11-18'
S1_VH_P5_END = '2021-06-01'

print('S2 export band order:', S2_PREDICTORS)
print('S1 export band order:', S1_PREDICTORS)


## 4. Water mask

In [ ]:
def get_jrc_water_mask(aoi, occurrence_threshold=JRC_OCCURRENCE_THRESHOLD):
    return (
        ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
        .select('occurrence')
        .gte(occurrence_threshold)
        .rename('water_mask')
        .clip(aoi)
    )


def apply_optional_water_mask_to_predictor_image(predictor_img, aoi):
    if APPLY_JRC_WATER_MASK:
        return predictor_img.updateMask(get_jrc_water_mask(aoi, JRC_OCCURRENCE_THRESHOLD))
    return predictor_img


def get_coverage_denominator_mask(aoi):
    if VALID_COVERAGE_DENOMINATOR.lower() == 'water':
        return get_jrc_water_mask(aoi, JRC_OCCURRENCE_THRESHOLD).selfMask().rename('denominator')
    if VALID_COVERAGE_DENOMINATOR.lower() == 'aoi':
        return ee.Image.constant(1).clip(aoi).selfMask().rename('denominator')
    raise ValueError("VALID_COVERAGE_DENOMINATOR must be 'water' or 'aoi'.")


print('JRC water mask applied before export:', APPLY_JRC_WATER_MASK)


## 5. Sentinel-2 predictor functions

In [ ]:
def mask_s2_sr(img):
    scl = img.select('SCL')
    mask = (
        scl.neq(3)    # cloud shadow
        .And(scl.neq(8))    # medium probability cloud
        .And(scl.neq(9))    # high probability cloud
        .And(scl.neq(10))   # thin cirrus
        .And(scl.neq(11))   # snow/ice
    )
    return img.updateMask(mask)


def add_s2_predictors(img):
    bands = img.select(
        ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8A', 'B8', 'B11', 'B12'],
        ['B',  'G',  'R',  'RE1', 'RE2', 'RE3', 'RE4', 'NIR', 'SWIR', 'SWIR2']
    ).toFloat()

    ndvi = bands.normalizedDifference(['NIR', 'R']).rename('NDVI')
    ndmi = bands.normalizedDifference(['NIR', 'SWIR']).rename('NDMI')
    mndwi = bands.normalizedDifference(['G', 'SWIR']).rename('MNDWI')

    awei = bands.expression(
        'B + 2.5 * G - 1.5 * (NIR + SWIR) - 0.25 * SWIR2',
        {
            'B': bands.select('B'),
            'G': bands.select('G'),
            'NIR': bands.select('NIR'),
            'SWIR': bands.select('SWIR'),
            'SWIR2': bands.select('SWIR2')
        }
    ).rename('AWEI')

    aweinsh = bands.expression(
        '4 * (G - SWIR) - (0.25 * NIR + 2.75 * SWIR2)',
        {
            'G': bands.select('G'),
            'NIR': bands.select('NIR'),
            'SWIR': bands.select('SWIR'),
            'SWIR2': bands.select('SWIR2')
        }
    ).rename('AWEInsh')

    return bands.addBands([ndvi, ndmi, mndwi, awei, aweinsh])


def get_s2_source_collection(start_date, end_date, aoi, cloud_pct=S2_CLOUD_PCT):
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct))
    )


def get_s2_collection(start_date, end_date, aoi, cloud_pct=S2_CLOUD_PCT):
    return (
        get_s2_source_collection(start_date, end_date, aoi, cloud_pct)
        .map(mask_s2_sr)
        .map(add_s2_predictors)
    )


def get_s2_awei_p95(aoi):
    return (
        get_s2_collection(S2_AWEI_P95_START, S2_AWEI_P95_END, aoi)
        .select('AWEI')
        .reduce(ee.Reducer.percentile([95]))
        .rename('AWEI_p95')
        .toFloat()
    )


S2_AWEI_P95_IMAGE = get_s2_awei_p95(winam)


def get_s2_predictor_image(start_date, end_date, aoi):
    # This is a one-day/acquisition-date snapshot, not a monthly composite.
    # Median is only used to combine overlapping same-date granules/tiles.
    collection = get_s2_collection(start_date, end_date, aoi)
    image = (
        collection
        .median()
        .addBands(S2_AWEI_P95_IMAGE)
        .select(S2_PREDICTORS)
        .clip(aoi)
    )
    return apply_optional_water_mask_to_predictor_image(image, aoi)


## 6. Sentinel-1 SCC predictor functions

In [ ]:
def add_s1_predictors_exact_scc(img, ref_angle_degrees=S1_REF_ANGLE_DEGREES):
    theta = img.select('angle').multiply(math.pi / 180.0)
    theta_ref = ee.Image.constant(ref_angle_degrees * math.pi / 180.0)

    correction_db = theta_ref.cos().pow(2).divide(theta.cos().pow(2)).log10().multiply(10.0)

    vh_corrected = img.select('VH').add(correction_db).rename('VH_corrected').toFloat()
    vv_corrected = img.select('VV').add(correction_db).rename('VV_corrected').toFloat()

    kernel_5x5 = ee.Kernel.square(radius=2, units='pixels', normalize=False)
    vh_smooth = vh_corrected.focal_median(kernel=kernel_5x5).rename('VH_smooth').toFloat()
    vv_smooth = vv_corrected.focal_median(kernel=kernel_5x5).rename('VV_smooth').toFloat()

    return img.addBands([vh_corrected, vv_corrected, vh_smooth, vv_smooth], overwrite=True)


def get_s1_source_collection(start_date, end_date, aoi):
    return (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    )


def get_s1_collection(start_date, end_date, aoi):
    return get_s1_source_collection(start_date, end_date, aoi).map(add_s1_predictors_exact_scc)


def get_s1_vh_p5(aoi):
    return (
        get_s1_collection(S1_VH_P5_START, S1_VH_P5_END, aoi)
        .select('VH_corrected')
        .reduce(ee.Reducer.percentile([5]))
        .rename('VH_p5')
        .toFloat()
    )


S1_VH_P5_IMAGE = get_s1_vh_p5(winam)


def get_s1_predictor_image(start_date, end_date, aoi):
    collection = get_s1_collection(start_date, end_date, aoi)
    image = (
        collection
        .median()
        .addBands(S1_VH_P5_IMAGE)
        .select(S1_PREDICTORS)
        .clip(aoi)
    )
    return apply_optional_water_mask_to_predictor_image(image, aoi)


## 7. Build the acquisition-date manifest

In [ ]:
def _add_date_property(img):
    return img.set('date_string', ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'))


def date_count_table(collection):
    hist = collection.map(_add_date_property).aggregate_histogram('date_string').getInfo()
    if not hist:
        return pd.DataFrame(columns=['date', 'image_count'])
    out = (
        pd.DataFrame({'date': list(hist.keys()), 'image_count': list(hist.values())})
        .sort_values('date')
        .reset_index(drop=True)
    )
    out['image_count'] = out['image_count'].astype(int)
    return out


def add_one_day_period_columns(df):
    df = df.copy()
    df['start_date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    df['end_date'] = (pd.to_datetime(df['date']) + pd.Timedelta(days=1)).dt.strftime('%Y-%m-%d')
    return df


def build_export_manifest():
    manifest_parts = []

    if EXPORT_S2:
        s2_source = get_s2_source_collection(S2_START_DATE, EXPORT_END_DATE, winam, S2_CLOUD_PCT)
        s2_dates = add_one_day_period_columns(date_count_table(s2_source))
        s2_dates['sensor'] = 'S2'
        s2_dates['prefix'] = s2_dates.apply(
            lambda r: f"winam_s2_predictors_{r['start_date']}_to_{r['end_date']}",
            axis=1
        )
        s2_dates['band_order'] = ','.join(S2_PREDICTORS)
        manifest_parts.append(s2_dates)
        print('S2 planned acquisition-date snapshot exports before coverage QA:', len(s2_dates))

    if EXPORT_S1:
        s1_source = get_s1_source_collection(S1_START_DATE, EXPORT_END_DATE, winam)
        s1_dates = add_one_day_period_columns(date_count_table(s1_source))
        s1_dates['sensor'] = 'S1'
        s1_dates['prefix'] = s1_dates.apply(
            lambda r: f"winam_s1_scc_predictors_{r['start_date']}_to_{r['end_date']}",
            axis=1
        )
        s1_dates['band_order'] = ','.join(S1_PREDICTORS)
        manifest_parts.append(s1_dates)
        print('S1 planned acquisition-date snapshot exports before coverage QA:', len(s1_dates))

    if not manifest_parts:
        return pd.DataFrame(columns=['manifest_row', 'sensor', 'date', 'start_date', 'end_date', 'image_count', 'prefix', 'band_order'])

    manifest = pd.concat(manifest_parts, ignore_index=True)
    manifest = manifest[['sensor', 'date', 'start_date', 'end_date', 'image_count', 'prefix', 'band_order']]
    manifest = manifest.sort_values(['start_date', 'sensor']).reset_index(drop=True)
    manifest.insert(0, 'manifest_row', range(len(manifest)))
    return manifest


manifest = build_export_manifest()
manifest_path = GEE_EXPORT_DIR / 'winam_snapshot_validated_predictor_export_manifest.csv'
manifest.to_csv(manifest_path, index=False)

print('Total planned snapshot rows before coverage QA:', len(manifest))
print('Saved manifest:', manifest_path)
display(manifest.head(10))
display(manifest.tail(10))


## 8. Progress file and pending export selection

In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def load_export_progress(progress_path=PROGRESS_PATH):
    progress_path = Path(progress_path)
    if not progress_path.exists():
        return {
            'created_at_utc': utc_now_iso(),
            'last_updated_utc': None,
            'last_processed_manifest_row': None,
            'last_processed_date': None,
            'last_processed_sensor': None,
            'last_processed_prefix': None,
            'last_processed_action': None,
            'started_exports_total': 0,
            'skipped_low_coverage_total': 0,
            'skipped_existing_total': 0,
        }

    with open(progress_path, 'r') as f:
        progress = json.load(f)

    # Backward-compatible fallback if you accidentally point this notebook
    # at an old progress JSON.
    if 'last_processed_manifest_row' not in progress:
        progress['last_processed_manifest_row'] = progress.get('last_started_manifest_row')
        progress['last_processed_date'] = progress.get('last_started_date')
        progress['last_processed_sensor'] = progress.get('last_started_sensor')
        progress['last_processed_prefix'] = progress.get('last_started_prefix')
        progress['last_processed_action'] = 'started_old_schema'

    return progress


def save_export_progress(progress, progress_path=PROGRESS_PATH):
    progress = dict(progress)
    progress['last_updated_utc'] = utc_now_iso()
    with open(progress_path, 'w') as f:
        json.dump(progress, f, indent=2)
    return progress


def export_prefix_exists(prefix, export_dir=GEE_EXPORT_DIR):
    export_dir = Path(export_dir)
    patterns = [
        f'{prefix}',
        f'{prefix}.tif', f'{prefix}.TIF',
        f'{prefix}.tiff', f'{prefix}.TIFF',
        f'{prefix}-*', f'{prefix}-*.tif', f'{prefix}-*.TIF',
        f'{prefix}-*.tiff', f'{prefix}-*.TIFF',
    ]
    for pattern in patterns:
        if any(p.is_file() for p in export_dir.glob(pattern)):
            return True
    return False


if RESET_EXPORT_PROGRESS and PROGRESS_PATH.exists():
    PROGRESS_PATH.unlink()
    print('Deleted progress file:', PROGRESS_PATH)

progress = load_export_progress(PROGRESS_PATH)

manifest_for_export = manifest.copy()
manifest_for_export['already_in_drive'] = False
manifest_for_export['already_processed_by_progress'] = False

if SKIP_PREFIXES_ALREADY_IN_DRIVE:
    manifest_for_export['already_in_drive'] = manifest_for_export['prefix'].apply(export_prefix_exists)

last_processed_manifest_row = progress.get('last_processed_manifest_row')
if (
    AUTO_RESUME_FROM_PROGRESS
    and SKIP_PREFIXES_ALREADY_PROCESSED_IN_PROGRESS
    and last_processed_manifest_row is not None
):
    last_processed_manifest_row = int(last_processed_manifest_row)
    manifest_for_export['already_processed_by_progress'] = (
        manifest_for_export['manifest_row'] <= last_processed_manifest_row
    )

if MANUAL_START_AT_MANIFEST_ROW is not None:
    manifest_for_export = manifest_for_export.loc[
        manifest_for_export['manifest_row'] >= int(MANUAL_START_AT_MANIFEST_ROW)
    ].copy()

pending_manifest = (
    manifest_for_export
    .loc[
        ~manifest_for_export['already_in_drive']
        & ~manifest_for_export['already_processed_by_progress']
    ]
    .reset_index(drop=True)
)

pending_path = GEE_EXPORT_DIR / 'winam_snapshot_validated_predictor_pending_manifest.csv'
pending_manifest.to_csv(pending_path, index=False)

print('Current progress:')
print(json.dumps(progress, indent=2))
print('\nTotal planned snapshot rows:', len(manifest))
print('Already exported in Drive:', int(manifest_for_export['already_in_drive'].sum()))
print('Skipped because already processed by progress:', int(manifest_for_export['already_processed_by_progress'].sum()))
print('Pending rows to QA/queue:', len(pending_manifest))
print('Saved pending manifest:', pending_path)

if len(pending_manifest):
    print('\nNext rows to QA/queue:')
    display(pending_manifest.head(min(20, len(pending_manifest))))
else:
    print('\nNo pending rows to QA/queue under the current resume/skip settings.')


## 9. Valid-pixel coverage QA helpers

In [ ]:
def get_valid_all_band_mask(image, band_order):
    '''
    Returns a binary image where every predictor band has valid data.
    Important: call this before the final export unmask(-9999).
    '''
    return (
        image
        .select(band_order)
        .mask()
        .reduce(ee.Reducer.min())
        .rename('valid_all_bands')
        .gt(0)
    )


def get_valid_pixel_stats(image, band_order, aoi):
    '''
    Counts valid all-band pixels and valid coverage fraction over the selected
    denominator mask, usually the JRC Winam water mask.
    '''
    valid_all_bands = get_valid_all_band_mask(image, band_order)
    denominator = get_coverage_denominator_mask(aoi)

    # 1 inside denominator where all bands valid, 0 elsewhere.
    valid_binary = (
        valid_all_bands
        .updateMask(denominator)
        .unmask(0)
        .rename('valid_all_bands')
    )

    valid_pixels = ee.Number(
        valid_binary.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=EXPORT_SCALE,
            maxPixels=1e13,
            tileScale=4
        ).get('valid_all_bands')
    )

    denominator_pixels = ee.Number(
        denominator.reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=aoi,
            scale=EXPORT_SCALE,
            maxPixels=1e13,
            tileScale=4
        ).get('denominator')
    )

    valid_fraction = ee.Number(
        ee.Algorithms.If(
            denominator_pixels.gt(0),
            valid_pixels.divide(denominator_pixels),
            0
        )
    )

    return ee.Dictionary({
        'valid_pixels': valid_pixels,
        'denominator_pixels': denominator_pixels,
        'valid_fraction': valid_fraction
    })


def thresholds_for_sensor(sensor):
    sensor = str(sensor).upper()
    if sensor == 'S2':
        return S2_MIN_VALID_PIXELS, S2_MIN_VALID_FRACTION
    if sensor == 'S1':
        return S1_MIN_VALID_PIXELS, S1_MIN_VALID_FRACTION
    raise ValueError(f'Unknown sensor: {sensor}')


def coverage_passes(sensor, stats):
    min_pixels, min_fraction = thresholds_for_sensor(sensor)
    valid_pixels = float(stats['valid_pixels'])
    valid_fraction = float(stats['valid_fraction'])
    return (valid_pixels >= min_pixels) and (valid_fraction >= min_fraction)


def format_stats(stats):
    return (
        f"valid_pixels={float(stats['valid_pixels']):,.0f}, "
        f"denominator_pixels={float(stats['denominator_pixels']):,.0f}, "
        f"valid_fraction={float(stats['valid_fraction']):.2%}"
    )


## 10. Export helpers

In [ ]:
def active_task_count(states=('READY', 'RUNNING')):
    tasks = ee.batch.Task.list()
    return sum(1 for task in tasks if task.status().get('state') in states)


def print_recent_task_status(limit=30):
    tasks = ee.batch.Task.list()[:limit]
    if not tasks:
        print('No Earth Engine tasks found.')
        return
    for task in tasks:
        status = task.status()
        print(f"{status.get('state'):>10} | {status.get('description')}")
        if 'error_message' in status:
            print('           Error:', status['error_message'])


def export_predictor_geotiff(image, description, file_prefix, band_order, aoi):
    # Do not unmask until after coverage QA has passed.
    # Here, we fill masked pixels with -9999 and tag that as GeoTIFF NoData.
    export_img = (
        image
        .select(band_order)
        .unmask(PREDICTOR_NODATA_VALUE)
        .toFloat()
    )

    export_kwargs = dict(
        image=export_img,
        description=description,
        folder=EE_EXPORT_FOLDER,
        fileNamePrefix=file_prefix,
        region=aoi,
        scale=EXPORT_SCALE,
        crs=EXPORT_CRS,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={
            'noData': PREDICTOR_NODATA_VALUE,
            'cloudOptimized': True
        }
    )

    if FILE_DIMENSIONS is not None:
        export_kwargs['fileDimensions'] = FILE_DIMENSIONS
    if SKIP_EMPTY_TILES is not None:
        export_kwargs['skipEmptyTiles'] = SKIP_EMPTY_TILES

    task = ee.batch.Export.image.toDrive(**export_kwargs)
    task.start()
    print(f'Started export: {description}')
    return task


def build_predictor_image_for_manifest_row(row):
    if row['sensor'] == 'S2':
        return get_s2_predictor_image(row['start_date'], row['end_date'], winam), S2_PREDICTORS
    if row['sensor'] == 'S1':
        return get_s1_predictor_image(row['start_date'], row['end_date'], winam), S1_PREDICTORS
    raise ValueError(f"Unknown sensor: {row['sensor']}")


def update_progress_after_event(row, action, progress):
    row = dict(row)
    progress = dict(progress)

    progress['last_processed_manifest_row'] = int(row['manifest_row'])
    progress['last_processed_date'] = row['date']
    progress['last_processed_sensor'] = row['sensor']
    progress['last_processed_prefix'] = row['prefix']
    progress['last_processed_action'] = action

    if action == 'started':
        progress['started_exports_total'] = int(progress.get('started_exports_total') or 0) + 1
    elif action == 'skipped_low_coverage':
        progress['skipped_low_coverage_total'] = int(progress.get('skipped_low_coverage_total') or 0) + 1
    elif action == 'skipped_existing':
        progress['skipped_existing_total'] = int(progress.get('skipped_existing_total') or 0) + 1

    progress = save_export_progress(progress, PROGRESS_PATH)
    return progress


def append_export_event_log(row, action, stats=None, task=None, reason=None, log_path=EXPORT_EVENT_LOG_PATH):
    row = dict(row)
    status = task.status() if task is not None else {}

    record = {
        'event_at_utc': utc_now_iso(),
        'action': action,
        'reason': reason,
        'manifest_row': int(row['manifest_row']),
        'sensor': row['sensor'],
        'date': row['date'],
        'start_date': row['start_date'],
        'end_date': row['end_date'],
        'image_count': int(row['image_count']),
        'prefix': row['prefix'],
        'valid_pixels': None if stats is None else float(stats['valid_pixels']),
        'denominator_pixels': None if stats is None else float(stats['denominator_pixels']),
        'valid_fraction': None if stats is None else float(stats['valid_fraction']),
        'task_id': status.get('id'),
        'task_state_at_start': status.get('state'),
    }

    log_path = Path(log_path)
    write_header = not log_path.exists()
    with open(log_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(record.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(record)

    return record


## 11. Launch the next resumable, coverage-gated batch

In [ ]:
export_tasks = {}
event_records = []

if not RUN_EXPORTS:
    print('RUN_EXPORTS is False, so no Earth Engine tasks were started.')
    print('Set RUN_EXPORTS = True in the settings cell when you are ready to export.')
elif len(pending_manifest) == 0:
    print('No pending rows under the current skip/resume settings.')
else:
    batch = pending_manifest.head(MAX_EXPORTS_PER_RUN).copy()
    print(f'Attempting to QA/queue up to {len(batch)} rows from the pending manifest.')
    print('First pending manifest row:', int(batch.iloc[0]['manifest_row']))
    print('Last pending manifest row in this planned batch:', int(batch.iloc[-1]['manifest_row']))

    for _, row in batch.iterrows():
        current_active = active_task_count()
        if current_active >= MAX_ACTIVE_TASKS:
            print(f'Stopping because active task count is {current_active}, at/above MAX_ACTIVE_TASKS={MAX_ACTIVE_TASKS}.')
            break

        # Re-check Drive immediately before starting in case files appeared since pending manifest was built.
        if SKIP_PREFIXES_ALREADY_IN_DRIVE and export_prefix_exists(row['prefix']):
            print(f"Skipping existing Drive export: {row['prefix']}")
            progress = update_progress_after_event(row, 'skipped_existing', progress)
            event_records.append(
                append_export_event_log(
                    row,
                    action='skipped_existing',
                    reason='prefix_already_exists_in_drive'
                )
            )
            continue

        image, band_order = build_predictor_image_for_manifest_row(row)

        stats = None
        if VALIDATE_BEFORE_EXPORT:
            stats = get_valid_pixel_stats(image, band_order, winam).getInfo()
            print(f"QA {row['prefix']} | {format_stats(stats)}")

            if not coverage_passes(row['sensor'], stats):
                min_pixels, min_fraction = thresholds_for_sensor(row['sensor'])
                reason = (
                    f"below_threshold: requires >= {min_pixels:,} valid pixels "
                    f"and >= {min_fraction:.0%} valid coverage"
                )
                print(f"Skipping low-coverage {row['sensor']} export: {row['prefix']} | {reason}")
                progress = update_progress_after_event(row, 'skipped_low_coverage', progress)
                event_records.append(
                    append_export_event_log(
                        row,
                        action='skipped_low_coverage',
                        stats=stats,
                        reason=reason
                    )
                )
                continue

        description = row['prefix']
        task = export_predictor_geotiff(
            image=image,
            description=description,
            file_prefix=row['prefix'],
            band_order=band_order,
            aoi=winam
        )

        export_tasks[row['prefix']] = task
        progress = update_progress_after_event(row, 'started', progress)
        event_records.append(
            append_export_event_log(
                row,
                action='started',
                stats=stats,
                task=task,
                reason='coverage_passed' if stats is not None else 'coverage_validation_disabled'
            )
        )

    print('\nStarted tasks this run:', len(export_tasks))
    print('Events logged this run:', len(event_records))
    print('Updated progress file:', PROGRESS_PATH)
    print('Updated event log:', EXPORT_EVENT_LOG_PATH)

    if event_records:
        display(pd.DataFrame(event_records))

    print('\nRecent Earth Engine task status:')
    print_recent_task_status(limit=max(30, len(export_tasks)))


## 12. Optional: preview one pending/manifest row and its valid coverage

In [ ]:
# Change this to inspect a particular manifest row.
PREVIEW_MANIFEST_ROW = None

if PREVIEW_MANIFEST_ROW is None:
    if len(pending_manifest):
        preview_row = pending_manifest.iloc[0]
    elif len(manifest):
        preview_row = manifest.iloc[0]
    else:
        preview_row = None
else:
    matches = manifest.loc[manifest['manifest_row'] == int(PREVIEW_MANIFEST_ROW)]
    preview_row = None if matches.empty else matches.iloc[0]

if preview_row is None:
    print('No manifest row available for preview.')
else:
    preview_img, preview_bands = build_predictor_image_for_manifest_row(preview_row)
    preview_stats = get_valid_pixel_stats(preview_img, preview_bands, winam).getInfo()
    print('Preview row:', preview_row.to_dict())
    print('Coverage:', format_stats(preview_stats))
    print('Passes gate:', coverage_passes(preview_row['sensor'], preview_stats))

    Map = geemap.Map(center=[-0.25, 34.45], zoom=9)
    Map.addLayer(winam, {}, 'Winam Gulf AOI')

    denominator = get_coverage_denominator_mask(winam)
    valid_mask = get_valid_all_band_mask(preview_img, preview_bands).updateMask(denominator)

    if preview_row['sensor'] == 'S2':
        Map.addLayer(preview_img.select(['NIR', 'R', 'G']), {'min': 0, 'max': 5000}, f"S2 false colour {preview_row['date']}")
        Map.addLayer(preview_img.select('NDVI'), {'min': -0.2, 'max': 0.8}, 'S2 NDVI', shown=False)
        Map.addLayer(preview_img.select('NDMI'), {'min': -0.5, 'max': 1.0}, 'S2 NDMI', shown=False)
    else:
        Map.addLayer(preview_img.select('VH_corrected'), {'min': -25, 'max': 0}, f"S1 VH corrected {preview_row['date']}")
        Map.addLayer(preview_img.select('VH_smooth'), {'min': -25, 'max': 0}, 'S1 VH smooth', shown=False)

    Map.addLayer(denominator, {'min': 0, 'max': 1, 'palette': ['000000', '00FFFF']}, 'Coverage denominator', shown=False)
    Map.addLayer(valid_mask, {'min': 0, 'max': 1, 'palette': ['FF00FF']}, 'Valid all-band pixels')
    display(Map)


## 13. Optional: summarise the event log

In [ ]:
if EXPORT_EVENT_LOG_PATH.exists():
    event_log = pd.read_csv(EXPORT_EVENT_LOG_PATH)
    print('Event log rows:', len(event_log))
    display(event_log.tail(20))

    print('\nAction counts:')
    display(event_log['action'].value_counts(dropna=False).to_frame('count'))

    if 'valid_fraction' in event_log.columns:
        print('\nValid fraction summary by sensor/action:')
        display(
            event_log
            .dropna(subset=['valid_fraction'])
            .groupby(['sensor', 'action'])['valid_fraction']
            .describe()
        )
else:
    print('No event log found yet:', EXPORT_EVENT_LOG_PATH)
